# 03 - Activation Steering and Locked Evaluation

[Open in Google Colab](https://colab.research.google.com/github/zhaoqyu/Lab-NLP/blob/mike/alignment_benchmark/notebooks/03_steer_and_evaluate.ipynb)

**Objective.** Select CAA interventions on KVS validation and score completed methods on locked KVS/AITA data.

Use a GPU runtime. Persistent artifacts are written to Google Drive, so interrupted Colab sessions can resume. Run cells from top to bottom.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Select a GPU runtime first.'


In [ ]:
REPO_URL = 'https://github.com/zhaoqyu/Lab-NLP.git'
BRANCH = 'mike'
REPO_DIR = Path('/content/Lab-NLP')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', 'alignment_benchmark/requirements-colab.txt'],
    check=True,
)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', './alignment_benchmark', '--no-deps'],
    check=True,
)


In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/Lab-NLP/valuebench-paper')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['VALUEBENCH_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
CONFIG = 'alignment_benchmark/configs/paper.yaml'

def run(*arguments: str) -> None:
    command = ['valuebench', *arguments, '--config', CONFIG]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

print('Persistent output:', OUTPUT_ROOT)


In [ ]:
run('doctor', '--strict')


## 1. Build one activation-steering intervention

Vectors use KVS train only. The layer and coefficient are selected on KVS validation only.

In [ ]:
TARGET = 'Security'
SITE = 'block'  # block or attn

run('build-steering', '--target', TARGET, '--site', SITE)


## 2. Evaluate the selected steering vector

In [ ]:
run(
    'evaluate',
    '--method', f'steering_{SITE}',
    '--target', TARGET,
    '--seed', '-1',
)


## 3. Evaluate one completed adapter intervention

Both the target and same-method control adapter must already be complete.

In [ ]:
RUN_ADAPTER_EVALUATION = False
METHOD = 'dpo'
ADAPTER_TARGET = 'Security'
SEED = 13

if RUN_ADAPTER_EVALUATION:
    run(
        'evaluate',
        '--method', METHOD,
        '--target', ADAPTER_TARGET,
        '--seed', str(SEED),
    )


## Queue-style evaluation

Once adapters or vectors exist, `run-next --phase evaluate` chooses one ready, unfinished evaluation.

In [ ]:
RUN_NEXT_READY_EVALUATION = False
if RUN_NEXT_READY_EVALUATION:
    run('run-next', '--phase', 'evaluate')

run('status')
